In [ ]:
# Required imports
import os
import subprocess

# Define target protein and the residue to center triangle attention on
PROT = "6KWC"
TRI_RESIDUE_IDX = 18

# Define all relevant directories
BASE_DATA_DIR = "/storage/ice1/shared/d-pace_community/alphafold/alphafold_2.3.2_data"
ROOT_DIR = os.path.expanduser("~/scratch")

# Local paths for saving results (these probably can remain unchanged)
ATTN_MAP_DIR = f"{ROOT_DIR}/vizfold-foundation/outputs/attention_files_{PROT}_demo_tri_{TRI_RESIDUE_IDX}"
ALIGNMENT_DIR = f"{ROOT_DIR}/vizfold-foundation/examples/monomer/alignments"
OUTPUT_DIR = f"{ROOT_DIR}/vizfold-foundation/outputs/my_outputs_align_{PROT}_demo_tri_{TRI_RESIDUE_IDX}"
IMAGE_OUTPUT_DIR = f"{ROOT_DIR}/vizfold-foundation/outputs/attention_images_{PROT}_demo_tri_{TRI_RESIDUE_IDX}"
FASTA_DIR = f"{ROOT_DIR}/vizfold-foundation/examples/monomer/fasta_dir_{PROT}"

os.makedirs(FASTA_DIR, exist_ok=True)
os.makedirs(f"{ROOT_DIR}/vizfold-foundation/outputs", exist_ok=True)

# Note: If this is a new protein, the ALIGNMENT_DIR does not need to be specified here or in the next cell
# In this case, the code will compute MSAs and alignments, which can take several hours

In [ ]:
%%bash
# Activate conda environment with OpenFold installed
source "$(conda info --base)/etc/profile.d/conda.sh"
export CONDA_ENVS_PATH=$CONDA_INSTALL_DIR/.conda/envs
export CONDA_PKGS_DIRS=$CONDA_INSTALL_DIR/.conda/pkgs
conda activate openfold_env

export MPLBACKEND=Agg
export PYTHONNOUSERSITE=1
cd ~/scratch/vizfold-foundation

# Define variables for the script
PROT="6KWC"
TRI_RESIDUE_IDX=18
BASE_DATA_DIR="/storage/ice1/shared/d-pace_community/alphafold/alphafold_2.3.2_data"
FASTA_DIR="$HOME/scratch/vizfold-foundation/examples/monomer/fasta_dir_${PROT}"
ALIGNMENT_DIR="$HOME/scratch/vizfold-foundation/examples/monomer/alignments"
OUTPUT_DIR="$HOME/scratch/vizfold-foundation/outputs/my_outputs_align_${PROT}_demo_tri_${TRI_RESIDUE_IDX}"
ATTN_MAP_DIR="$HOME/scratch/vizfold-foundation/outputs/attention_files_${PROT}_demo_tri_${TRI_RESIDUE_IDX}"

# Run OpenFold inference
python3 run_pretrained_openfold.py \
    ${FASTA_DIR} \
    ${BASE_DATA_DIR}/pdb_mmcif/mmcif_files \
    --use_precomputed_alignments ${ALIGNMENT_DIR} \
    --output_dir ${OUTPUT_DIR} \
    --config_preset model_1_ptm \
    --uniref90_database_path ${BASE_DATA_DIR}/uniref90/uniref90.fasta \
    --mgnify_database_path ${BASE_DATA_DIR}/mgnify/mgy_clusters_2022_05.fa \
    --pdb70_database_path ${BASE_DATA_DIR}/pdb70/pdb70 \
    --uniclust30_database_path ${BASE_DATA_DIR}/uniclust30/uniclust30_2018_08 \
    --bfd_database_path ${BASE_DATA_DIR}/bfd/bfd_metaclust_clu_complete_id30_c90_final_seq.sorted_opt \
    --save_outputs \
    --model_device "cuda:0" \
    --attn_map_dir ${ATTN_MAP_DIR} \
    --num_recycles_save 1 \
    --triangle_residue_idx ${TRI_RESIDUE_IDX} \
    --demo_attn

echo "Inference complete. Outputs saved to ${OUTPUT_DIR} and attention maps saved to ${ATTN_MAP_DIR}."

In [ ]:
# Render predicted 3D structure and save as PNG image
from visualize_attention_general_utils import render_pdb_to_image

PDB_FILE = os.path.join(OUTPUT_DIR, f"predictions/{PROT}_1_model_1_ptm_relaxed.pdb")
FNAME = f"predicted_structure_{PROT}_tri_{TRI_RESIDUE_IDX}.png"

render_pdb_to_image(PDB_FILE, IMAGE_OUTPUT_DIR, FNAME)

In [ ]:
# Import visualization utilities
from visualize_attention_3d_demo_utils import plot_pymol_attention_heads
from visualize_attention_arc_diagram_demo_utils import generate_arc_diagrams, parse_fasta_sequence

# Setup visualization output directories
PROT = "6KWC"
TRI_RESIDUE_IDX = 18
LAYER_IDX = 47 # selected layer for attention evaluation
TOP_K = 50 # show top-k attention links (limit to 500)

BASE_DIR = os.path.expanduser("~/scratch/vizfold-foundation")
OUTPUT_DIR = os.path.join(BASE_DIR, f"outputs/my_outputs_align_{PROT}_demo_tri_{TRI_RESIDUE_IDX}")
ATTN_MAP_DIR = os.path.join(BASE_DIR, f"outputs/attention_files_{PROT}_demo_tri_{TRI_RESIDUE_IDX}")
IMAGE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "images")
PDB_FILE = os.path.join(OUTPUT_DIR, f"predictions/{PROT}_1_model_1_ptm_relaxed.pdb")
FASTA_PATH = os.path.join(BASE_DIR, f"examples/monomer/fasta_dir_{PROT}/{PROT}.fasta")

# Setup output sub-directories and create if they don't exist
output_dir_msa = os.path.join(IMAGE_OUTPUT_DIR, 'msa_row_attention_plots')
output_dir_tri = os.path.join(IMAGE_OUTPUT_DIR, 'tri_start_attention_plots')
os.makedirs(output_dir_msa, exist_ok=True)
os.makedirs(output_dir_tri, exist_ok=True)

# Generate 3D attention plots for MSA row attention
plot_pymol_attention_heads(
    pdb_file=PDB_FILE,
    attention_dir=ATTN_MAP_DIR,
    output_dir=output_dir_msa,
    protein=PROT,
    attention_type="msa_row",
    top_k=TOP_K,
    layer_idx=LAYER_IDX
)

# Generate 3D attention plots for triangle start attention
plot_pymol_attention_heads(
    pdb_file=PDB_FILE,
    attention_dir=ATTN_MAP_DIR,
    output_dir=output_dir_tri,
    protein=PROT,
    attention_type="triangle_start",
    residue_indices=[TRI_RESIDUE_IDX],
    top_k=TOP_K,
    layer_idx=LAYER_IDX
)

# Parse FASTA for arc diagrams
residue_seq = parse_fasta_sequence(FASTA_PATH)

# Generate arc diagrams for MSA row attention
generate_arc_diagrams(
    attention_dir=ATTN_MAP_DIR,
    residue_sequence=residue_seq,
    output_dir=output_dir_msa,
    protein=PROT,
    attention_type="msa_row",
    top_k=TOP_K,
    layer_idx=LAYER_IDX
)

# Generate arc diagrams for triangle start attention
generate_arc_diagrams(
    attention_dir=ATTN_MAP_DIR,
    residue_sequence=residue_seq,
    output_dir=output_dir_tri,
    protein=PROT,
    attention_type="triangle_start",
    residue_indices=[TRI_RESIDUE_IDX],
    top_k=TOP_K,
    layer_idx=LAYER_IDX
)


In [ ]:
# Import function for combining attention plots
from visualize_attention_general_utils import generate_combined_attention_panels

# Combine MSA row plots
generate_combined_attention_panels(
    attention_type="msa_row",
    protein=PROT,
    layer_idx=LAYER_IDX,
    output_dir_3d=output_dir_msa,
    output_dir_arc=output_dir_msa,
    combined_output_dir=IMAGE_OUTPUT_DIR,
)

# Combine triangle start plots
generate_combined_attention_panels(
    attention_type="triangle_start",
    protein=PROT,
    layer_idx=LAYER_IDX,
    output_dir_3d=output_dir_tri,
    output_dir_arc=output_dir_tri,
    combined_output_dir=IMAGE_OUTPUT_DIR,
    residue_indices=[TRI_RESIDUE_IDX]
)
